# EXIOBASE3 hotspot analysis for Sweden with pymrio

This notebook loads the EXIOBASE3 monetary PxP IOT, calculates the missing MRIO tables with `pymrio`, and produces a Sweden-focused hotspot analysis for:

1. economic importance,
2. greenhouse-gas impacts,
3. material footprint,

for both consumption-based accounting (CBA) and production-based accounting (PBA). It also derives source-country shares for the hotspot sectors.

## Key assumptions used in this notebook

- **Economic importance** is represented by the `factor_inputs` extension row **`Value Added`**.
- **GHG impacts** are represented by the `impact` extension row **`global warming (GWP100)`** only when a characterized `impact` extension is actually available. If not, the notebook skips GHG with an explicit note rather than assuming a proxy.
- **Material footprint** is represented from the **`materials`** extension. The notebook first tries to find a single “total used-material” style row. If none is found, it sums eligible material rows after checking that the units are consistent.
- **Source-country shares for PBA sectors are trivial by definition**: for Swedish PBA hotspots, the source country is Sweden (`SE`) because PBA records impacts occurring in Swedish production sectors.
- The country-origin decomposition for **CBA** is implemented in a memory-conscious way from `S`, `L`, and Swedish final demand, rather than by fully diagonalizing a stressor, because that approach becomes very memory-intensive for full EXIOBASE systems.

## Before running

- Adjust the settings in the configuration cell if needed.
- Expect parsing and `calc_all()` on EXIOBASE to take noticeable time and memory.

## Changelog relative to v8

1. **Data-integrity guardrail.** A new cell (inserted right after the extension
   inspection cell) scans the resolved emissions extension for known
   data-corruption patterns. It flags with severity `ERROR`:

   - Core combustion rows (`CO2 / CH4 / N2O - combustion - air`) that are zero
     across every region of S. This was observed in the 2024 EXIOBASE3 pxp
     parse and silently understates fossil-combustion GHG by about 50% for
     Sweden.
   - Any single raw F row whose global GWP100-weighted total exceeds a
     plausibility ceiling. In the 2024 parse, `CH4 - agriculture - air` is
     inflated roughly 40x real-world and `N2O - agriculture - air` roughly
     15x, which in turn explains the implausibly large 2024 CBA.

   If any `ERROR` finding is present and `GHG_FAIL_ON_DATA_INTEGRITY_ERROR=True`
   (the default), the notebook skips the GHG indicator entirely for that run.
   Set the toggle to `False` if you want to see results anyway (for example
   while investigating the flag itself).

2. **Per-row diagnostic alignment fix.** `ghg_proxy_row_diagnostics` and
   `ghg_core_row_check_table` previously routed each single-row proxy through
   `manual_cba_sector_vector` and `manual_pba_sector_vector`. In the 2024 parse
   that path returned `0.0` for every per-row CBA even though the aggregate
   ranking path returned 790 Gt CO2-eq. v9 replaces both functions with versions
   that do explicit `.reindex(mrio.L.index).fillna(0.0)` onto numpy arrays, so
   per-row numbers are now bit-consistent with the aggregate ranking.

Nothing else in the notebook is changed. All other helpers, config knobs,
ranking logic, country decomposition logic, and output formats are identical
to v8.

## Recommended workflow for the Stockholm project

Based on the data-integrity investigation, I recommend using **two different
EXIOBASE3 releases for different indicators**:

- **Materials and value added**: use the 2024 release. These indicators are
  unaffected by the GHG corruption in that parse and give you the most recent
  economic snapshot.
- **GHG**: fall back to the 2022 release. Its characterized impact extension
  produces internally consistent totals (around 51 Mt CO2-eq PBA and 90 Mt CBA
  for Sweden, CBA/PBA ratio ~1.75, top CBA sector Construction) that align with
  Swedish EPA reference figures.

Mixing reference years across indicators is a trade-off and should be flagged
explicitly in the Stockholm report. The alternative (using 2022 for all three
indicators) costs you two years of economic development in the materials and
value-added pictures, which are the pieces the hybrid Stockholm method most
depends on. That alternative is easy to switch to by setting `DATA_PATH` in
the config cell to the 2022 zip and re-running.


In [ ]:
# Optional: install missing packages from inside the notebook
# %pip install pymrio pandas numpy matplotlib openpyxl

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pymrio

warnings.filterwarnings("ignore", category=FutureWarning)

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

# -----------------------------
# User configuration
# -----------------------------
DATA_PATH = Path(r"C:\EXIOBASE3\IOT_2024_pxp.zip")
TARGET_REGION = "SE"

TOP_N_SECTORS = 15               # ranking tables and charts
TOP_N_SOURCE_SECTORS = 5         # hotspot sectors per indicator/account to trace to countries
TOP_N_SOURCE_COUNTRIES = 10      # number of source countries shown per CBA hotspot sector
TOP_N_DESTINATION_COUNTRIES = 10 # number of destination countries shown per PBA hotspot sector

# Set this to a row label or list of row labels if you want to override
# the automatic material-footprint row selection.
# Examples:
# MATERIAL_ROW_SELECTION = "Total material extraction"
# MATERIAL_ROW_SELECTION = [("Used", "Biomass"), ("Used", "Metal ores")]
MATERIAL_ROW_SELECTION = None

# Optional manual override for value added rows when the factor_inputs extension
# does not provide a single "Value Added" total row.
VALUE_ADDED_ROW_SELECTION = None

# GHG proxy configuration used only when no characterized `impact` extension is present.
# Default factors follow common 100-year GWPs for CO2, CH4, and N2O.
GWP100_FACTORS = {
    "CO2": 1.0,
    "CH4": 28.0,
    "N2O": 265.0,
}

# Set to True if you explicitly want to include *_bio rows in the raw-emissions GHG proxy.
GHG_INCLUDE_BIOGENIC = False

# Include rows that are already reported in CO2-equivalent units (for example HFC/PFC).
GHG_INCLUDE_PRECHARACTERIZED_EQ_ROWS = True

# Include SF6 if present in physical mass units and characterize it with the configured factor.
GHG_INCLUDE_SF6 = True

# Core rows that should normally not be zero for a complete production-based fossil/combustion profile.
# These are used only for diagnostics/warnings.
GHG_CORE_DIAGNOSTIC_ROWS = [
    "CO2 - combustion - air",
    "CH4 - combustion - air",
    "N2O - combustion - air",
]

# Optional manual override for GHG rows when using the raw air-emissions extension.
# Example format:
# GHG_ROW_SELECTION = {
#     ("CO2", "combustion", "air"): 1.0,
#     ("CH4", "combustion", "air"): 28.0,
#     ("N2O", "combustion", "air"): 265.0,
# }
GHG_ROW_SELECTION = None


# Choose which GHG definition to use when both are available.
# "impact" = characterized impact extension (preferred when present)
# "proxy"  = raw-emissions proxy built from selected CO2/CH4/N2O rows
GHG_PREFERRED_SOURCE = "impact"

# When an impact extension is available and raw emissions are also available,
# build the proxy as well and compare the two definitions.
GHG_COMPARE_PROXY_WHEN_IMPACT_AVAILABLE = True

# Set to True to skip GHG results entirely unless a characterized impact
# extension is available. Useful when you want strict cross-year comparability.
GHG_FAIL_IF_ONLY_PROXY = False

# Heuristic warning threshold for the total GHG CBA/PBA ratio.
GHG_SUSPICIOUS_CBA_TO_PBA_RATIO = 5.0


# v9: data-integrity guardrail behaviour.
# If True, and the guardrail detects ERROR-level findings in the emissions
# extension (zero combustion rows or implausible magnitude), the notebook skips
# the raw-emissions GHG proxy and produces no GHG ranking for that run.
# If False, GHG results are produced anyway. Treat them as exploratory only.
GHG_FAIL_ON_DATA_INTEGRITY_ERROR = True

OUTPUT_DIR = Path.cwd() / "exiobase3_sweden_hotspot_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)



In [ ]:
from types import SimpleNamespace


def row_to_text(row_label):
    """Render row labels consistently for exact matching and reporting."""
    if isinstance(row_label, tuple):
        return " | ".join(str(x) for x in row_label)
    return str(row_label)


def as_listlike(row_selector):
    if isinstance(row_selector, (list, tuple, pd.Index, np.ndarray)):
        return row_selector
    return [row_selector]


def select_rows(df, row_selector):
    """
    Safe row selection for regular and MultiIndex row labels.
    Returns a DataFrame even for a single row.
    """
    if isinstance(row_selector, tuple) and not isinstance(df.index, pd.MultiIndex):
        return df.loc[[row_selector]]
    if isinstance(row_selector, (list, pd.Index, np.ndarray)):
        return df.loc[list(row_selector)]
    return df.loc[[row_selector]]


def sum_selected_rows(df, row_selector):
    selected = select_rows(df, row_selector)
    if selected.shape[0] == 1:
        return selected.iloc[0]
    return selected.sum(axis=0)


def get_level_key(index_like, preferred_name, fallback_position):
    if isinstance(index_like, pd.MultiIndex):
        if preferred_name in (index_like.names or []):
            return preferred_name
        return fallback_position
    raise ValueError("Expected a MultiIndex.")


def split_region_sector_columns(df, target_region):
    """
    Return the sector columns for a specific region from a standard pymrio account table.
    """
    if isinstance(df.columns, pd.MultiIndex):
        level = get_level_key(df.columns, "region", 0)
        return df.xs(target_region, axis=1, level=level)
    raise ValueError("Expected a pymrio-style DataFrame with MultiIndex columns (region, sector).")


def aggregate_final_demand_by_region(Y):
    """
    Aggregate final demand categories to one column per region.
    """
    if isinstance(Y.columns, pd.MultiIndex):
        col_level = get_level_key(Y.columns, "region", 0)
        return Y.T.groupby(level=col_level, sort=False).sum().T
    return Y.copy()


def ranking_from_account(account_df, row_selector, target_region, account_name, indicator_name, unit=None):
    """
    Build a tidy ranking table for one indicator and one account type.
    """
    regional_df = split_region_sector_columns(account_df, target_region)
    values = sum_selected_rows(regional_df, row_selector).astype(float)
    values = values.sort_values(ascending=False)

    total = values.sum()
    out = (
        values.rename("value")
        .reset_index()
        .rename(columns={"index": "sector", "sector": "sector"})
    )
    if "sector" not in out.columns:
        out.columns = ["sector", "value"]

    out["account"] = account_name
    out["indicator"] = indicator_name
    out["region"] = target_region
    out["share_of_region_total"] = np.where(total != 0, out["value"] / total, np.nan)
    out["rank"] = np.arange(1, len(out) + 1)
    out["unit"] = unit
    return out[["account", "indicator", "region", "sector", "value", "share_of_region_total", "rank", "unit"]]


def _extract_units_from_selection(unit_obj, row_selector):
    if unit_obj is None:
        return []

    if isinstance(unit_obj, pd.DataFrame):
        selected = select_rows(unit_obj, row_selector)
        if selected.shape[1] == 1:
            units = selected.iloc[:, 0].astype(str).str.strip().dropna().unique().tolist()
        else:
            units = pd.unique(selected.astype(str).stack().str.strip()).tolist()
    elif isinstance(unit_obj, pd.Series):
        if isinstance(row_selector, (list, pd.Index, np.ndarray)):
            selected = unit_obj.loc[list(row_selector)]
        else:
            selected = unit_obj.loc[[row_selector]]
        units = selected.astype(str).str.strip().dropna().unique().tolist()
    else:
        return []

    return [u for u in units if u and u.lower() != "nan"]


def detect_common_unit(ext, row_selector):
    """
    Try to detect a common unit across selected extension rows.
    """
    try:
        units = _extract_units_from_selection(getattr(ext, "unit", None), row_selector)
        if len(units) == 1:
            return units[0]
        if len(units) == 0:
            return None
        raise ValueError(f"Selected rows do not have a single common unit: {units}")
    except Exception:
        return None


def get_extension_map(mrio):
    return {name: getattr(mrio, name) for name in mrio.get_extensions() if hasattr(mrio, name)}


def resolve_extension(mrio, exact=None, contains=None, row_contains=None, required=False, label="extension"):
    """
    Resolve an extension dynamically from the parsed MRIO object.
    """
    ext_map = get_extension_map(mrio)

    for candidate in (exact or []):
        if candidate in ext_map:
            return ext_map[candidate], candidate

    contains = [c.lower() for c in (contains or [])]
    if contains:
        for name, ext in ext_map.items():
            lname = name.lower()
            if any(c in lname for c in contains):
                return ext, name

    row_contains = [c.lower() for c in (row_contains or [])]
    if row_contains:
        for name, ext in ext_map.items():
            try:
                rows = [row_to_text(r).lower() for r in list(ext.get_rows())[:500]]
            except Exception:
                continue
            if any(any(token in txt for token in row_contains) for txt in rows):
                return ext, name

    if required:
        raise KeyError(
            f"Could not resolve {label}. Available extensions: {list(ext_map.keys())}"
        )
    return None, None


def preview_extension_rows(ext, n=50):
    try:
        return [row_to_text(x) for x in list(ext.get_rows())[:n]]
    except Exception:
        return []


def choose_first_matching_row(ext, exact=None, contains=None):
    """
    Return the first matching extension row using exact and substring matching.
    """
    rows = list(ext.get_rows())
    row_map = {row_to_text(r).strip().lower(): r for r in rows}

    for candidate in (exact or []):
        key = str(candidate).strip().lower()
        if key in row_map:
            return row_map[key]

    for needle in (contains or []):
        needle = str(needle).strip().lower()
        for r in rows:
            txt = row_to_text(r).strip().lower()
            if needle in txt:
                return r

    available_preview = [row_to_text(r) for r in rows[:30]]
    raise KeyError(
        "Could not find a matching row. "
        f"Exact candidates={exact}, contains candidates={contains}. "
        f"First available rows: {available_preview}"
    )


def find_rows_containing(ext, include_any, exclude_any=None):
    matches = []
    include_any = [s.lower() for s in (include_any or [])]
    exclude_any = [s.lower() for s in (exclude_any or [])]
    for row in list(ext.get_rows()):
        txt = row_to_text(row).lower()
        if include_any and not any(token in txt for token in include_any):
            continue
        if exclude_any and any(token in txt for token in exclude_any):
            continue
        matches.append(row)
    return matches


def choose_value_added_rows(factor_inputs_ext, override=None):
    """
    Resolve rows representing gross value added.
    Prefer an explicit total row; otherwise sum the component rows.
    """
    if override is not None:
        return override, "User-specified value-added row selection"

    try:
        total_row = choose_first_matching_row(
            factor_inputs_ext,
            exact=["Value Added", "value added"],
            contains=["value added"],
        )
        return total_row, f"Automatically selected value-added total row: {row_to_text(total_row)}"
    except Exception:
        pass

    component_rows = find_rows_containing(
        factor_inputs_ext,
        include_any=[
            "taxes less subsidies",
            "other net taxes",
            "compensation of employees",
            "operating surplus",
            "mixed income",
        ],
        exclude_any=[],
    )

    if component_rows:
        return component_rows, "No single value-added total row found; summing value-added component rows"

    all_rows = list(factor_inputs_ext.get_rows())
    return all_rows, "No explicit value-added row pattern found; summing all factor-input rows"



def choose_material_rows(materials_ext, override=None):
    """
    Heuristic material-row selector.

    Strategy:
    1. Use explicit override if provided.
    2. Prefer an explicit total used-material row when it exists.
    3. Otherwise sum rows starting with 'Domestic Extraction Used -'.
    4. Exclude rows that look like totals when detailed rows are selected, to avoid double counting.
    """
    if override is not None:
        return override, "User-specified material row selection"

    rows = list(materials_ext.get_rows())
    row_text = pd.Series({row: row_to_text(row).lower() for row in rows})

    exact_total_candidates = []
    for row, txt in row_text.items():
        if (
            "domestic extraction used" in txt
            and (
                txt.strip() == "domestic extraction used"
                or txt.endswith(" - total")
                or " total " in txt
                or txt.startswith("total ")
            )
        ):
            exact_total_candidates.append(row)

    if len(exact_total_candidates) == 1:
        return exact_total_candidates[0], f"Automatically selected material total row: {row_to_text(exact_total_candidates[0])}"

    used_rows = []
    for row, txt in row_text.items():
        if not txt.startswith("domestic extraction used"):
            continue
        if "unused" in txt:
            continue
        if txt.strip() == "domestic extraction used":
            continue
        if "total" in txt:
            continue
        used_rows.append(row)

    if len(used_rows) > 0:
        return used_rows, "Automatically selected detailed 'Domestic Extraction Used' rows and summed them"

    unit = detect_common_unit(materials_ext, rows)
    if unit is not None:
        return rows, "No clear material-total row found; summing all material rows because units appear consistent"

    raise ValueError(
        "Could not determine a safe material-footprint row selection automatically. "
        "Inspect the material extension rows and set MATERIAL_ROW_SELECTION explicitly."
    )



def choose_ghg_row_factors(
    emissions_ext,
    override=None,
    default_factors=None,
    include_biogenic=False,
    include_precharacterized_eq_rows=True,
    include_sf6=True,
):
    """
    Build a transparent GHG row-to-factor mapping from the air-emissions extension.

    Included by default:
    - CO2, CH4, N2O rows in physical mass units
    - HFC/PFC rows already reported in CO2-eq units (factor 1)
    - SF6 if present (using the configured factor)

    Excluded by default:
    - biogenic rows when include_biogenic=False
    - rows whose text suggests they are already aggregated impacts (e.g. GWP / CO2-equivalent)
      for CO2/CH4/N2O matching
    """
    if override is not None:
        return override, "User-specified GHG row selection"

    default_factors = default_factors or {"CO2": 1.0, "CH4": 28.0, "N2O": 265.0}
    row_factor_map = {}
    notes = []

    gas_patterns = {
        "CO2": ["carbon dioxide", "co2"],
        "CH4": ["methane", "ch4"],
        "N2O": ["nitrous oxide", "n2o"],
    }

    gas_excludes = {
        "CO2": ["co2-equivalent", "co2 equivalent", "kg co2-eq", "gwp"],
        "CH4": ["co2-equivalent", "co2 equivalent", "kg co2-eq", "gwp"],
        "N2O": ["nitrogen oxides", "nox", "co2-equivalent", "co2 equivalent", "kg co2-eq", "gwp"],
    }

    if not include_biogenic:
        for gas in gas_excludes:
            gas_excludes[gas] = list(gas_excludes[gas]) + ["_bio", " biogenic "]

    # First: physical-mass CO2/CH4/N2O rows
    for gas, patterns in gas_patterns.items():
        rows = find_rows_containing(emissions_ext, include_any=patterns, exclude_any=gas_excludes.get(gas))
        matched = []
        for row in rows:
            unit = str(detect_common_unit(emissions_ext, row) or "").lower()
            # Keep physical rows; skip already-equivalent rows here to avoid double counting
            if "co2-eq" in unit or "co2 eq" in unit:
                continue
            row_factor_map[row] = float(default_factors[gas])
            matched.append(row)
        if matched:
            notes.append(f"{gas}: {len(matched)} physical row(s)")
        else:
            notes.append(f"{gas}: no physical row found")

    # Second: already-characterized HFC/PFC rows
    if include_precharacterized_eq_rows:
        for gas in ["HFC", "PFC"]:
            rows = find_rows_containing(emissions_ext, include_any=[gas.lower()], exclude_any=[])
            matched = []
            for row in rows:
                unit = str(detect_common_unit(emissions_ext, row) or "").lower()
                if "co2-eq" in unit or "co2 eq" in unit:
                    row_factor_map[row] = 1.0
                    matched.append(row)
            if matched:
                notes.append(f"{gas}: {len(matched)} CO2-eq row(s) included with factor 1")
            else:
                notes.append(f"{gas}: no CO2-eq row found")

    # Third: SF6 physical rows
    if include_sf6:
        rows = find_rows_containing(emissions_ext, include_any=["sf6"], exclude_any=[])
        matched = []
        for row in rows:
            unit = str(detect_common_unit(emissions_ext, row) or "").lower()
            if "co2-eq" in unit or "co2 eq" in unit:
                row_factor_map[row] = 1.0
                matched.append(row)
            elif "SF6" in default_factors:
                row_factor_map[row] = float(default_factors["SF6"])
                matched.append(row)
        if matched:
            notes.append(f"SF6: {len(matched)} row(s) included")
        else:
            notes.append("SF6: no row found")

    if not row_factor_map:
        preview = preview_extension_rows(emissions_ext, n=80)
        raise ValueError(
            "Could not identify any configured GHG rows in the emissions extension. "
            f"First rows: {preview}"
        )

    if not include_biogenic:
        notes.append("biogenic rows excluded by configuration")

    return row_factor_map, "Automatically built raw-emissions GHG proxy using " + ", ".join(notes)


def characterize_account_from_row_factors(account_df, row_factor_map, row_name):
    pieces = []
    for row, factor in row_factor_map.items():
        piece = select_rows(account_df, row).astype(float) * float(factor)
        pieces.append(piece)
    combined = pd.concat(pieces, axis=0).groupby(level=0).sum() if len(pieces) > 1 else pieces[0]
    if isinstance(combined, pd.DataFrame) and combined.shape[0] > 1:
        combined = combined.sum(axis=0)
    elif isinstance(combined, pd.DataFrame) and combined.shape[0] == 1:
        combined = combined.iloc[0]
    return pd.DataFrame([combined], index=pd.Index([row_name], name="impact"))



def build_characterized_proxy_extension(ext, row_factor_map, row_name="GHG proxy", impact_unit="kg CO2-eq"):
    proxy = SimpleNamespace()
    proxy.name = row_name
    proxy.S = characterize_account_from_row_factors(ext.S, row_factor_map, row_name)
    proxy.unit = pd.DataFrame({"unit": [impact_unit]}, index=pd.Index([row_name], name="impact"))
    proxy.D_cba = None
    proxy.D_pba = None
    return proxy


def as_series(vector_like):
    if isinstance(vector_like, pd.DataFrame):
        if vector_like.shape[1] == 1:
            return vector_like.iloc[:, 0].astype(float)
        raise ValueError("Expected a Series-like object or a single-column DataFrame.")
    if isinstance(vector_like, pd.Series):
        return vector_like.astype(float)
    return pd.Series(vector_like, dtype=float)


def get_output_vector(mrio):
    if hasattr(mrio, "x") and mrio.x is not None:
        return as_series(mrio.x)
    return as_series(mrio.L.dot(mrio.Y.sum(axis=1)))


def selected_S_vector(ext, row_selector):
    return sum_selected_rows(ext.S, row_selector).astype(float)


def manual_cba_sector_vector(mrio, ext, row_selector, target_region, Y_agg):
    """
    Compute sector-specific CBA directly from S, L, and aggregated Y.

    For one destination region, multiply the multiplier vector M = S @ L
    with the destination-region final-demand column and group by sector.
    """
    S_vec = selected_S_vector(ext, row_selector)
    M_vec = as_series(S_vec.dot(mrio.L))
    y_reg = as_series(Y_agg[target_region])

    if not M_vec.index.equals(y_reg.index):
        y_reg = y_reg.reindex(M_vec.index)

    sector_level = get_level_key(M_vec.index, "sector", 1)
    sector_values = (M_vec * y_reg).groupby(level=sector_level).sum().sort_values(ascending=False)
    return sector_values.astype(float)


def manual_pba_sector_vector(mrio, ext, row_selector, target_region):
    """
    Compute sector-specific PBA directly from S and x.
    """
    S_vec = selected_S_vector(ext, row_selector)
    x_vec = get_output_vector(mrio)

    if not S_vec.index.equals(x_vec.index):
        x_vec = x_vec.reindex(S_vec.index)

    direct = (S_vec * x_vec).astype(float)
    region_level = get_level_key(direct.index, "region", 0)
    sector_level = get_level_key(direct.index, "sector", 1)
    region_direct = direct.xs(target_region, level=region_level)
    if isinstance(region_direct.index, pd.MultiIndex):
        region_direct = region_direct.groupby(level=sector_level).sum()
    region_direct = region_direct.sort_values(ascending=False)
    return region_direct.astype(float)


def manual_account_value(mrio, ext, row_selector, account_name, target_region, target_sector, Y_agg):
    if account_name == "CBA":
        vec = manual_cba_sector_vector(mrio, ext, row_selector, target_region, Y_agg)
    elif account_name == "PBA":
        vec = manual_pba_sector_vector(mrio, ext, row_selector, target_region)
    else:
        raise ValueError("account_name must be 'CBA' or 'PBA'")
    return float(vec.loc[target_sector])


def ranking_from_manual_account(mrio, ext, row_selector, target_region, account_name, indicator_name, Y_agg, unit=None):
    """
    Build a tidy ranking table using manual S/L/Y definitions instead of relying on stored D_cba/D_pba.
    """
    if account_name == "CBA":
        values = manual_cba_sector_vector(mrio, ext, row_selector, target_region, Y_agg)
    elif account_name == "PBA":
        values = manual_pba_sector_vector(mrio, ext, row_selector, target_region)
    else:
        raise ValueError("account_name must be 'CBA' or 'PBA'")

    total = values.sum()
    out = values.rename("value").reset_index()
    if "sector" not in out.columns:
        out.columns = ["sector", "value"]

    out["account"] = account_name
    out["indicator"] = indicator_name
    out["region"] = target_region
    out["share_of_region_total"] = np.where(total != 0, out["value"] / total, np.nan)
    out["rank"] = np.arange(1, len(out) + 1)
    out["unit"] = unit
    return out[["account", "indicator", "region", "sector", "value", "share_of_region_total", "rank", "unit"]]


def summarize_tail_with_other(df, country_col, value_col, share_col, total_value, top_n, force_include=None):
    """
    Keep the largest countries, optionally force-include specific ones, and add an OTHER row for the remainder.
    """
    work = df.copy()
    force_include = force_include or []
    work = work.sort_values(value_col, ascending=False).reset_index(drop=True)

    if top_n is None or top_n >= len(work):
        return work.reset_index(drop=True)

    forced = work[work[country_col].isin(force_include)]
    top = work.head(top_n)
    kept = pd.concat([top, forced], ignore_index=True).drop_duplicates(subset=[country_col]).reset_index(drop=True)

    kept_value = float(kept[value_col].sum())
    other_value = float(total_value - kept_value)

    if abs(other_value) > 1e-12:
        other_row = {col: kept.iloc[0][col] if len(kept) else np.nan for col in kept.columns}
        other_row[country_col] = "OTHER"
        other_row[value_col] = other_value
        other_row[share_col] = other_value / total_value if total_value != 0 else np.nan
        kept = pd.concat([kept, pd.DataFrame([other_row])], ignore_index=True)

    return kept.reset_index(drop=True)


def compare_extension_and_manual_accounts(mrio, ext, row_selector, target_region, Y_agg, indicator_name):
    rows = []

    # manual accounts
    cba_manual = manual_cba_sector_vector(mrio, ext, row_selector, target_region, Y_agg)
    pba_manual = manual_pba_sector_vector(mrio, ext, row_selector, target_region)

    # extension accounts if available
    if getattr(ext, "D_cba", None) is not None:
        try:
            cba_ext = sum_selected_rows(split_region_sector_columns(ext.D_cba, target_region), row_selector).astype(float)
            cba_ext = cba_ext.reindex(cba_manual.index)
            rows.append({
                "indicator": indicator_name,
                "account": "CBA",
                "manual_total": float(cba_manual.sum()),
                "extension_total": float(cba_ext.sum()),
                "difference": float(cba_manual.sum() - cba_ext.sum()),
                "relative_difference": float((cba_manual.sum() - cba_ext.sum()) / cba_manual.sum()) if float(cba_manual.sum()) != 0 else np.nan,
            })
        except Exception:
            pass

    if getattr(ext, "D_pba", None) is not None:
        try:
            pba_ext = sum_selected_rows(split_region_sector_columns(ext.D_pba, target_region), row_selector).astype(float)
            pba_ext = pba_ext.reindex(pba_manual.index)
            rows.append({
                "indicator": indicator_name,
                "account": "PBA",
                "manual_total": float(pba_manual.sum()),
                "extension_total": float(pba_ext.sum()),
                "difference": float(pba_manual.sum() - pba_ext.sum()),
                "relative_difference": float((pba_manual.sum() - pba_ext.sum()) / pba_manual.sum()) if float(pba_manual.sum()) != 0 else np.nan,
            })
        except Exception:
            pass

    return pd.DataFrame(rows)



def build_final_demand_vector_for_region_sector(Y_agg, target_region, target_sector):
    """
    Build the sector-specific final-demand vector that matches pymrio.calc_accounts'
    diagonalization logic: all source regions supplying one sector to one destination region.
    """
    if not isinstance(Y_agg.index, pd.MultiIndex):
        raise ValueError("Expected Y_agg rows to be a MultiIndex of (region, sector).")

    sector_level = get_level_key(Y_agg.index, "sector", 1)
    y_target = pd.Series(0.0, index=Y_agg.index, dtype=float)

    mask = Y_agg.index.get_level_values(sector_level) == target_sector
    y_target.loc[mask] = Y_agg.loc[mask, target_region].astype(float).to_numpy()
    return y_target


def extract_account_value(account_df, row_selector, target_region, target_sector):
    values = sum_selected_rows(account_df, row_selector).astype(float)
    return float(values.loc[(target_region, target_sector)])



def cba_source_country_shares(mrio, ext, row_selector, target_region, target_sector, Y_agg, top_n=10):
    """
    Attribute the CBA of one (target_region, target_sector) column to source countries.
    """
    y_target = build_final_demand_vector_for_region_sector(Y_agg, target_region, target_sector)
    x_source = as_series(mrio.L.dot(y_target)).rename("required_output")
    selected_S = selected_S_vector(ext, row_selector)

    if not selected_S.index.equals(x_source.index):
        x_source = x_source.reindex(selected_S.index)

    source_vector = (selected_S * x_source).rename("source_value")
    region_level = get_level_key(source_vector.index, "region", 0)

    country_totals = source_vector.groupby(level=region_level).sum().sort_values(ascending=False)
    total = float(country_totals.sum())

    full = country_totals.reset_index()
    full.columns = ["source_country", "source_value"]
    full["source_share"] = np.where(total != 0, full["source_value"] / total, np.nan)
    full["target_region"] = target_region
    full["target_sector"] = target_sector
    full["model_total"] = total
    full["residual_to_model_total"] = full["source_value"].sum() - total
    full = full[[
        "target_region",
        "target_sector",
        "source_country",
        "source_value",
        "source_share",
        "model_total",
        "residual_to_model_total",
    ]]

    shown = summarize_tail_with_other(
        full,
        country_col="source_country",
        value_col="source_value",
        share_col="source_share",
        total_value=total,
        top_n=top_n,
        force_include=[target_region],
    )
    shown["target_region"] = target_region
    shown["target_sector"] = target_sector
    shown["model_total"] = total
    shown["residual_to_model_total"] = shown["source_value"].sum() - total
    shown = shown[[
        "target_region",
        "target_sector",
        "source_country",
        "source_value",
        "source_share",
        "model_total",
        "residual_to_model_total",
    ]]

    return full.reset_index(drop=True), shown.reset_index(drop=True)



def pba_destination_country_shares(mrio, ext, row_selector, source_region, source_sector, Y_agg, top_n=10):
    """
    Allocate the direct PBA of one production sector across destination countries'
    final demand using the output decomposition x = L @ y_country.

    Important: this is a *final-demand destination* allocation, not a direct export-
    partner table. A Swedish sector can therefore have foreign destination shares even
    when its immediate exports are limited, because the sector can supply upstream
    value chains that ultimately satisfy final demand abroad.
    """
    if not isinstance(Y_agg.columns, pd.Index):
        raise ValueError("Expected Y_agg columns to be a single Index of destination regions.")

    output_by_destination = mrio.L.dot(Y_agg.astype(float))
    source_output_by_destination = as_series(output_by_destination.loc[(source_region, source_sector)])

    selected_S = selected_S_vector(ext, row_selector)
    direct_coeff = float(selected_S.loc[(source_region, source_sector)])

    destination_values = (source_output_by_destination * direct_coeff).sort_values(ascending=False)
    total = float(destination_values.sum())

    full = destination_values.rename("destination_value").reset_index()
    full.columns = ["destination_country", "destination_value"]
    full["destination_share"] = np.where(total != 0, full["destination_value"] / total, np.nan)
    full["source_region"] = source_region
    full["source_sector"] = source_sector
    full["model_total"] = total
    full["residual_to_model_total"] = full["destination_value"].sum() - total
    full = full[[
        "source_region",
        "source_sector",
        "destination_country",
        "destination_value",
        "destination_share",
        "model_total",
        "residual_to_model_total",
    ]]

    shown = summarize_tail_with_other(
        full,
        country_col="destination_country",
        value_col="destination_value",
        share_col="destination_share",
        total_value=total,
        top_n=top_n,
        force_include=[source_region],
    )
    shown["source_region"] = source_region
    shown["source_sector"] = source_sector
    shown["model_total"] = total
    shown["residual_to_model_total"] = shown["destination_value"].sum() - total
    shown = shown[[
        "source_region",
        "source_sector",
        "destination_country",
        "destination_value",
        "destination_share",
        "model_total",
        "residual_to_model_total",
    ]]

    return full.reset_index(drop=True), shown.reset_index(drop=True)



def summarise_breakdown_quality(
    df,
    group_cols,
    value_col,
    total_col,
    reference_total_col=None,
    share_col=None,
    atol=1e-6,
    rtol=1e-9,
):
    """
    Summarise whether a country breakdown closes back to the model total.

    Parameters
    ----------
    df : DataFrame
        Full (not truncated) breakdown table.
    group_cols : list[str]
        Columns identifying one hotspot sector.
    value_col : str
        Contribution value column to sum.
    total_col : str
        Column containing the model total repeated on each row.
    reference_total_col : str, optional
        Additional reference total to compare against, e.g. the sector value shown
        in the ranking table.
    share_col : str, optional
        If provided, also report the summed shares.
    atol, rtol : float
        Absolute and relative tolerances for pass/fail flags.
    """
    agg_spec = {
        "reconstructed_total": (value_col, "sum"),
        "model_total": (total_col, "first"),
        "min_contribution": (value_col, "min"),
        "max_contribution": (value_col, "max"),
        "negative_contribution_count": (value_col, lambda s: int((pd.Series(s) < 0).sum())),
    }
    if reference_total_col is not None and reference_total_col in df.columns:
        agg_spec["reference_total"] = (reference_total_col, "first")
    if share_col is not None and share_col in df.columns:
        agg_spec["share_sum"] = (share_col, "sum")

    summary = (
        df.groupby(group_cols, dropna=False)
        .agg(**agg_spec)
        .reset_index()
    )

    summary["abs_difference"] = summary["reconstructed_total"] - summary["model_total"]
    summary["relative_difference"] = np.where(
        summary["model_total"] != 0,
        summary["abs_difference"] / summary["model_total"],
        np.nan,
    )
    summary["passes_model_total_check"] = (
        summary["abs_difference"].abs()
        <= (atol + rtol * summary["model_total"].abs())
    )

    if "reference_total" in summary.columns:
        summary["difference_to_reference"] = summary["reconstructed_total"] - summary["reference_total"]
        summary["relative_difference_to_reference"] = np.where(
            summary["reference_total"] != 0,
            summary["difference_to_reference"] / summary["reference_total"],
            np.nan,
        )
        summary["passes_reference_check"] = (
            summary["difference_to_reference"].abs()
            <= (atol + rtol * summary["reference_total"].abs())
        )

    if "share_sum" in summary.columns:
        summary["share_sum_difference_from_1"] = summary["share_sum"] - 1.0
        summary["passes_share_sum_check"] = summary["share_sum_difference_from_1"].abs() <= (10 * rtol)

    return summary



def plot_top_sectors(df, title, top_n=15):
    """
    Simple horizontal bar chart for the top sectors.
    """
    plot_df = df.head(top_n).sort_values("value", ascending=True)

    plt.figure(figsize=(10, max(5, 0.35 * len(plot_df))))
    plt.barh(plot_df["sector"], plot_df["value"])
    plt.title(title)
    plt.xlabel(plot_df["unit"].dropna().iloc[0] if "unit" in df.columns and df["unit"].notna().any() else "value")
    plt.tight_layout()
    plt.show()




def compare_two_sector_vectors(vec_a, vec_b, label_a="A", label_b="B", top_n=15):
    aligned = pd.concat(
        [vec_a.rename(label_a), vec_b.rename(label_b)],
        axis=1,
    ).fillna(0.0)
    aligned["difference"] = aligned[label_b] - aligned[label_a]
    aligned["ratio"] = np.where(aligned[label_a] != 0, aligned[label_b] / aligned[label_a], np.nan)

    top_overlap = len(set(vec_a.sort_values(ascending=False).head(top_n).index) & set(vec_b.sort_values(ascending=False).head(top_n).index))
    top_union = len(set(vec_a.sort_values(ascending=False).head(top_n).index) | set(vec_b.sort_values(ascending=False).head(top_n).index))
    spearman_like = aligned[label_a].rank(ascending=False, method="average").corr(
        aligned[label_b].rank(ascending=False, method="average"),
        method="pearson",
    )

    summary = pd.DataFrame([{
        "top_n": top_n,
        f"{label_a}_total": float(vec_a.sum()),
        f"{label_b}_total": float(vec_b.sum()),
        "difference_total": float(vec_b.sum() - vec_a.sum()),
        "ratio_total": float(vec_b.sum() / vec_a.sum()) if float(vec_a.sum()) != 0 else np.nan,
        "top_n_overlap_count": int(top_overlap),
        "top_n_jaccard": float(top_overlap / top_union) if top_union else np.nan,
        "rank_correlation_like_spearman": float(spearman_like) if pd.notna(spearman_like) else np.nan,
        f"top1_{label_a}": vec_a.sort_values(ascending=False).index[0] if len(vec_a) else None,
        f"top1_{label_b}": vec_b.sort_values(ascending=False).index[0] if len(vec_b) else None,
    }])

    details = (
        aligned
        .sort_values(label_b, ascending=False)
        .reset_index()
        .rename(columns={"index": "sector", "sector": "sector"})
    )
    if "sector" not in details.columns:
        details.columns = ["sector", label_a, label_b, "difference", "ratio"]
    return summary, details



def _aligned_row_totals_for_ghg_diag(mrio, emissions_ext, raw_row, factor, target_region, Y_agg):
    """
    Compute (PBA_global, CBA_target_region) for a SINGLE raw emission row with its
    GWP factor, using explicit index alignment against mrio.L.index. This bypasses the
    silent-zero alignment failure that occurs in some pymrio parses when a single-row
    proxy extension is passed back through manual_cba_sector_vector /
    manual_pba_sector_vector.

    Notes
    -----
    v9 change: previous versions of ghg_proxy_row_diagnostics built a single-row
    proxy extension and routed it through the manual_* functions. In the 2024
    EXIOBASE3 parse, that path returned 0.0 for every per-row CBA even when the
    aggregate proxy CBA was nonzero, because pandas dot-product silently produced
    a misaligned result that then broadcast to NaN when multiplied by y_target.
    Here we explicitly reindex the row-slice onto mrio.L.index before any
    dot-product / multiply, and use numpy-level arithmetic for the final totals,
    which matches what the aggregate ranking path computes end-to-end.
    """
    # Pull the single raw row out of ext.S as a Series, multiply by the GWP factor,
    # and align to mrio.L.index so the downstream dot product cannot silently drift.
    S_row = select_rows(emissions_ext.S, raw_row).iloc[0].astype(float) * float(factor)
    S_row_aligned = S_row.reindex(mrio.L.index).fillna(0.0)

    # PBA global: S * x summed across all (region, sector).
    x_vec = as_series(get_output_vector(mrio)).reindex(mrio.L.index).fillna(0.0)
    pba_global = float((S_row_aligned.values * x_vec.values).sum())

    # CBA for target_region: S @ L @ y_target.
    y_target = as_series(Y_agg[target_region]).reindex(mrio.L.index).fillna(0.0)
    M_row = S_row_aligned.values @ mrio.L.values           # shape (N,)
    cba_target = float((M_row * y_target.values).sum())

    return pba_global, cba_target


def ghg_proxy_row_diagnostics(mrio, emissions_ext, row_factor_map, target_region, Y_agg):
    """
    Row-level diagnostics aligned with the main ranking logic.

    v9 change: now uses numpy-level, explicitly-reindexed arithmetic (see
    `_aligned_row_totals_for_ghg_diag`) instead of routing each single-row proxy
    through manual_cba_sector_vector / manual_pba_sector_vector. This fixes the
    silent-zero per-row CBA that appeared with the 2024 EXIOBASE parse while the
    aggregate ranking path still returned a nonzero (and in that dataset,
    implausibly large) CBA total.

    The `cba_total_weighted` column is now the Swedish CBA contribution of each
    raw emission row; `pba_total_weighted` is the global (not Swedish) direct
    stressor total weighted by the GWP factor, kept for comparability with
    per-row global magnitudes quoted in EXIOBASE documentation.
    """
    rows = []
    for raw_row, factor in row_factor_map.items():
        pba_total, cba_total = _aligned_row_totals_for_ghg_diag(
            mrio=mrio,
            emissions_ext=emissions_ext,
            raw_row=raw_row,
            factor=factor,
            target_region=target_region,
            Y_agg=Y_agg,
        )
        rows.append({
            "raw_row": row_to_text(raw_row),
            "factor": float(factor),
            "cba_total_weighted": cba_total,
            "pba_total_weighted": pba_total,
        })
    out = pd.DataFrame(rows)
    if not out.empty:
        out["cba_share_of_proxy_total"] = np.where(
            out["cba_total_weighted"].sum() != 0,
            out["cba_total_weighted"] / out["cba_total_weighted"].sum(),
            np.nan,
        )
        out["pba_share_of_proxy_total"] = np.where(
            out["pba_total_weighted"].sum() != 0,
            out["pba_total_weighted"] / out["pba_total_weighted"].sum(),
            np.nan,
        )
        out = out.sort_values("cba_total_weighted", ascending=False).reset_index(drop=True)
    return out


def ghg_data_integrity_guardrail(
    emissions_ext,
    core_combustion_rows=None,
    plausibility_thresholds_kg_co2eq=None,
    gwp_factors=None,
):
    """
    Scan the raw air-emissions extension for known data-corruption patterns that
    would silently inflate or deflate GHG CBA/PBA results for any region.

    Checks performed:

    1. Presence check for core combustion rows (CO2 / CH4 / N2O - combustion - air).
       These should normally exist in any complete EXIOBASE air-emissions extension.

    2. Zero-across-all-regions check for each core combustion row. If a row is
       globally zero (observed in the 2024 EXIOBASE pxp parse), any production-based
       or consumption-based fossil-combustion GHG result will be systematically
       understated by roughly the share of combustion in total GHG.

    3. Magnitude-plausibility check against simple global upper bounds. Rows that
       exceed the configured upper bound by GWP100 weighting are flagged as
       potentially corrupted (observed for CH4 - agriculture - air in the 2024
       EXIOBASE pxp parse, which was inflated roughly 40x over real-world global
       CH4 from agriculture).

    The function returns a diagnostic dataframe. It never raises. The calling code
    is responsible for deciding whether to proceed, skip the GHG indicator, or
    fall back to a different dataset.

    Parameters
    ----------
    emissions_ext : pymrio-style extension
        The air-emissions extension resolved earlier in the notebook.
    core_combustion_rows : list of str, optional
        Row labels expected to be populated. Defaults to the three CO2/CH4/N2O
        combustion-air rows.
    plausibility_thresholds_kg_co2eq : dict, optional
        Mapping row-label substring (lowercased) -> upper bound on the global
        GWP100-weighted total, in kg CO2-eq. Defaults cover the typical worst
        offenders observed in corrupted EXIOBASE releases.
    gwp_factors : dict, optional
        Gas -> GWP100 factor. Defaults to the same values used by the proxy
        builder.
    """
    core_combustion_rows = core_combustion_rows or [
        "CO2 - combustion - air",
        "CH4 - combustion - air",
        "N2O - combustion - air",
    ]
    gwp_factors = gwp_factors or {"CO2": 1.0, "CH4": 28.0, "N2O": 265.0}

    # Upper bounds in kg CO2-eq. Values are conservative: they allow for a factor
    # of ~2 over real-world totals, so they only trip on clear corruption. Units
    # are kg CO2-eq after GWP100 weighting, i.e. CH4 kg x 28 or N2O kg x 265.
    default_thresholds = {
        "ch4 - agriculture - air":        2.0e13,   # real ~4e12
        "ch4 - waste - air":              5.0e12,
        "n2o - agriculture - air":        1.5e13,   # real ~3e12
        "co2 - non combustion - cement":  5.0e12,
        "co2 - agriculture - peat decay": 5.0e12,
        "hfc - air":                      5.0e12,
        "pfc - air":                      5.0e11,
        "sf6 - air":                      5.0e11,
    }
    plausibility_thresholds_kg_co2eq = plausibility_thresholds_kg_co2eq or default_thresholds

    findings = []
    F_like = emissions_ext.F if hasattr(emissions_ext, "F") and emissions_ext.F is not None else None

    # Check 1 & 2: presence and zero-across-all-regions for core combustion rows.
    for label in core_combustion_rows:
        try:
            row_vals = select_rows(emissions_ext.S, label).iloc[0]
            present = True
            s_total = float(row_vals.astype(float).abs().sum())
            f_total = None
            if F_like is not None:
                try:
                    f_vals = select_rows(F_like, label).iloc[0]
                    f_total = float(f_vals.astype(float).abs().sum())
                except Exception:
                    f_total = None
            zero_s = s_total == 0.0
            zero_f = (f_total == 0.0) if f_total is not None else None
            severity = "ERROR" if zero_s else "OK"
            message = (
                "Core combustion row has all-zero S across every (region, sector). "
                "Fossil-combustion CBA and PBA will be systematically understated."
                if zero_s else "Core combustion row is populated in S."
            )
        except Exception:
            present = False
            s_total = np.nan
            f_total = np.nan
            zero_s = None
            zero_f = None
            severity = "ERROR"
            message = "Core combustion row not found in the emissions extension."

        findings.append({
            "check": "core_combustion_row",
            "row": label,
            "severity": severity,
            "present": present,
            "s_abs_sum": s_total,
            "f_abs_sum": f_total,
            "s_zero": zero_s,
            "f_zero": zero_f,
            "message": message,
        })

    # Check 3: magnitude plausibility on the F matrix if present, otherwise skip.
    # We match each threshold substring against all rows and report any exceedances.
    if F_like is not None:
        # Build a quick lookup of (row_label) -> gwp factor based on gas token.
        def _gwp_for_label(label):
            ll = str(label).lower()
            if ll.startswith("co2") or " co2" in ll: return gwp_factors.get("CO2", 1.0)
            if ll.startswith("ch4") or " ch4" in ll or "methane" in ll: return gwp_factors.get("CH4", 28.0)
            if ll.startswith("n2o") or " n2o" in ll or "nitrous oxide" in ll: return gwp_factors.get("N2O", 265.0)
            # Already-weighted rows in most EXIOBASE releases are HFC/PFC/SF6 stored in kg CO2-eq.
            if "hfc" in ll or "pfc" in ll or "sf6" in ll: return 1.0
            return None

        for substring, upper_bound in plausibility_thresholds_kg_co2eq.items():
            matches = []
            for row_label in list(F_like.index):
                if substring.lower() not in row_to_text(row_label).lower():
                    continue
                matches.append(row_label)

            for row_label in matches:
                try:
                    vals = select_rows(F_like, row_label).iloc[0].astype(float)
                except Exception:
                    continue
                gwp = _gwp_for_label(row_label) or 1.0
                global_weighted = float(vals.sum() * gwp)
                over_bound = global_weighted > upper_bound
                severity = "ERROR" if over_bound else "OK"
                message = (
                    f"Global GWP-weighted total = {global_weighted:.3e} kg CO2-eq exceeds the "
                    f"plausibility ceiling of {upper_bound:.3e}. Row is likely corrupted in this EXIOBASE parse."
                    if over_bound else
                    f"Global GWP-weighted total = {global_weighted:.3e} kg CO2-eq is within the plausibility range."
                )
                findings.append({
                    "check": "magnitude_plausibility",
                    "row": row_to_text(row_label),
                    "severity": severity,
                    "present": True,
                    "s_abs_sum": None,
                    "f_abs_sum": float(vals.abs().sum()),
                    "s_zero": None,
                    "f_zero": None,
                    "message": message,
                })
    else:
        findings.append({
            "check": "magnitude_plausibility",
            "row": "(F matrix not exposed on the extension)",
            "severity": "WARN",
            "present": False,
            "s_abs_sum": None,
            "f_abs_sum": None,
            "s_zero": None,
            "f_zero": None,
            "message": "Extension does not expose an F matrix, so magnitude plausibility could not be checked.",
        })

    out = pd.DataFrame(findings)
    # Reorder columns for readability.
    col_order = ["check", "severity", "row", "present", "s_abs_sum", "f_abs_sum", "s_zero", "f_zero", "message"]
    out = out[[c for c in col_order if c in out.columns]]
    return out


def ghg_core_row_check_table(mrio, emissions_ext, row_factor_map, target_region, Y_agg, core_rows=None):
    """
    Diagnostic check for key rows that should usually carry non-zero Swedish GHG totals.

    v9 change: same alignment fix as ghg_proxy_row_diagnostics. See
    `_aligned_row_totals_for_ghg_diag` for details.
    """
    core_rows = core_rows or []
    diagnostics = []
    available_rows = {row_to_text(r): r for r in list(row_factor_map.keys())}

    for label in core_rows:
        raw_row = available_rows.get(label)
        if raw_row is None:
            diagnostics.append({
                "raw_row": label,
                "selected_in_proxy": False,
                "factor": np.nan,
                "cba_total_weighted": np.nan,
                "pba_total_weighted": np.nan,
                "both_zero": np.nan,
                "warning": "row not selected in proxy",
            })
            continue

        factor = row_factor_map[raw_row]
        pba_total, cba_total = _aligned_row_totals_for_ghg_diag(
            mrio=mrio,
            emissions_ext=emissions_ext,
            raw_row=raw_row,
            factor=factor,
            target_region=target_region,
            Y_agg=Y_agg,
        )
        diagnostics.append({
            "raw_row": label,
            "selected_in_proxy": True,
            "factor": float(factor),
            "cba_total_weighted": cba_total,
            "pba_total_weighted": pba_total,
            "both_zero": bool(abs(cba_total) == 0 and abs(pba_total) == 0),
            "warning": "selected but zero in both CBA and PBA" if (abs(cba_total) == 0 and abs(pba_total) == 0) else "",
        })

    return pd.DataFrame(diagnostics)


def ghg_sanity_check_table(rankings, indicator_key="ghg", suspicious_ratio=5.0):
    subset = rankings.query("indicator == @indicator_key").copy()
    if subset.empty:
        return pd.DataFrame()
    totals = subset.groupby("account")["value"].sum().to_dict()
    cba_total = float(totals.get("CBA", np.nan))
    pba_total = float(totals.get("PBA", np.nan))
    ratio = cba_total / pba_total if pd.notna(cba_total) and pd.notna(pba_total) and pba_total != 0 else np.nan
    cba_top = subset.query("account == 'CBA'").sort_values("rank").head(1)
    pba_top = subset.query("account == 'PBA'").sort_values("rank").head(1)
    return pd.DataFrame([{
        "indicator": indicator_key,
        "cba_total": cba_total,
        "pba_total": pba_total,
        "cba_to_pba_ratio": ratio,
        "suspicious_ratio_threshold": suspicious_ratio,
        "ratio_exceeds_threshold": bool(pd.notna(ratio) and ratio > suspicious_ratio),
        "top_cba_sector": cba_top["sector"].iloc[0] if not cba_top.empty else None,
        "top_cba_sector_share": float(cba_top["share_of_region_total"].iloc[0]) if not cba_top.empty else np.nan,
        "top_pba_sector": pba_top["sector"].iloc[0] if not pba_top.empty else None,
        "top_pba_sector_share": float(pba_top["share_of_region_total"].iloc[0]) if not pba_top.empty else np.nan,
    }])



In [ ]:
# Check that the file exists before starting heavy calculations
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Could not find EXIOBASE at: {DATA_PATH}\n"
        "Update DATA_PATH in the configuration cell to match your local file."
    )

print(f"Loading EXIOBASE from: {DATA_PATH}")
mrio = pymrio.parse_exiobase3(DATA_PATH)

print("Calculating missing MRIO tables and accounts. This can take substantial time and memory.")
mrio.calc_all()

print("Loaded regions:", len(mrio.get_regions()))
print("Loaded sectors:", len(mrio.get_sectors()))
print("Extensions:", list(mrio.get_extensions()))

In [ ]:

# Basic checks and dynamic extension inspection
assert TARGET_REGION in mrio.get_regions(), f"{TARGET_REGION} is not a valid EXIOBASE region code."

ext_map = get_extension_map(mrio)
print("Available extensions:")
display(pd.DataFrame({"extension": list(ext_map.keys())}))

factor_inputs_ext, factor_inputs_name = resolve_extension(
    mrio,
    exact=["factor_inputs", "factor_input"],
    contains=["factor_input"],
    row_contains=["value added"],
    required=True,
    label="factor-input extension",
)

materials_ext, materials_name = resolve_extension(
    mrio,
    exact=["materials", "material"],
    contains=["material", "resource"],
    row_contains=["biomass", "metal ores", "non-metallic minerals", "domestic extraction", "material"],
    required=False,
    label="material extension",
)

impact_ext, impact_name = resolve_extension(
    mrio,
    exact=["impact", "impacts"],
    contains=["impact"],
    row_contains=["global warming", "gwp100"],
    required=False,
    label="impact extension",
)

emissions_ext, emissions_name = resolve_extension(
    mrio,
    exact=["air_emissions", "emissions", "satellite"],
    contains=["air_emission", "emission", "satellite"],
    row_contains=["carbon dioxide", "methane", "nitrous oxide", "co2", "ch4", "n2o"],
    required=False,
    label="emissions extension",
)

print(f"Resolved factor-input extension: {factor_inputs_name}")
if materials_ext is not None:
    print(f"Resolved material extension: {materials_name}")
else:
    print("Could not resolve a dedicated material extension automatically.")

if impact_ext is not None:
    print(f"Resolved impact extension: {impact_name}")
else:
    print("No characterized impact extension found.")

if emissions_ext is not None:
    print(f"Resolved emissions extension: {emissions_name}")
else:
    print("No raw emissions extension found.")

print("Factor input rows (first 30):")
display(pd.DataFrame({"factor_input_row": preview_extension_rows(factor_inputs_ext, 30)}))

if materials_ext is not None:
    print("Material extension rows (first 50):")
    display(pd.DataFrame({"material_row": preview_extension_rows(materials_ext, 50)}))

if impact_ext is not None:
    print("Impact rows (first 30):")
    display(pd.DataFrame({"impact_row": preview_extension_rows(impact_ext, 30)}))

if emissions_ext is not None:
    print("Emissions rows (first 50):")
    display(pd.DataFrame({"emission_row": preview_extension_rows(emissions_ext, 50)}))


In [ ]:
# v9 data-integrity guardrail for the emissions extension.
#
# Scans `emissions_ext` (if present) for two known corruption patterns:
#   - Core combustion rows with all-zero S across every region.
#   - F rows whose global GWP100-weighted total exceeds a plausibility ceiling.
#
# If any ERROR finding is present AND GHG_FAIL_ON_DATA_INTEGRITY_ERROR is True,
# the GHG indicator is skipped for this run. You can set the toggle to False in
# the configuration cell if you want to inspect results anyway.

GHG_FAIL_ON_DATA_INTEGRITY_ERROR = globals().get("GHG_FAIL_ON_DATA_INTEGRITY_ERROR", True)
ghg_data_integrity_findings = pd.DataFrame()
ghg_data_integrity_has_errors = False

if emissions_ext is not None:
    ghg_data_integrity_findings = ghg_data_integrity_guardrail(
        emissions_ext,
        core_combustion_rows=GHG_CORE_DIAGNOSTIC_ROWS,
        gwp_factors=GWP100_FACTORS,
    )
    print("GHG data-integrity guardrail findings:")
    display(ghg_data_integrity_findings)

    errors = ghg_data_integrity_findings.query("severity == 'ERROR'")
    ghg_data_integrity_has_errors = not errors.empty

    if ghg_data_integrity_has_errors:
        print(f"\nGuardrail detected {len(errors)} ERROR finding(s) in the emissions data.")
        if GHG_FAIL_ON_DATA_INTEGRITY_ERROR:
            print(
                "GHG_FAIL_ON_DATA_INTEGRITY_ERROR is True, so the GHG indicator will be skipped "
                "for this run. Rerun with GHG_FAIL_ON_DATA_INTEGRITY_ERROR=False to force results."
            )
        else:
            print(
                "GHG_FAIL_ON_DATA_INTEGRITY_ERROR is False, so GHG results will be produced anyway. "
                "Treat them with caution. See the findings above for what is flagged."
            )
    else:
        print("\nGuardrail: no data-integrity errors detected.")

    try:
        ghg_data_integrity_findings.to_csv(OUTPUT_DIR / "ghg_data_integrity_findings.csv", index=False)
    except Exception:
        pass
else:
    print("No emissions extension resolved; skipping data-integrity guardrail.")


In [ ]:
# Prepare the final-demand aggregation used by pymrio account calculations
Y_agg = aggregate_final_demand_by_region(mrio.Y)

print("Y aggregated to one column per region:")
display(Y_agg.iloc[:5, :5])

print("Selected region:", TARGET_REGION)

In [ ]:
# v9: honour the data-integrity guardrail outcome.
# If the emissions data has ERROR-level findings and the user has not set the override,
# skip the GHG proxy path entirely. The characterized impact extension is still usable
# if the guardrail only flagged raw-emissions rows.
if (
    globals().get("ghg_data_integrity_has_errors", False)
    and globals().get("GHG_FAIL_ON_DATA_INTEGRITY_ERROR", True)
):
    print(
        "v9: skipping raw-emissions GHG proxy because the data-integrity guardrail "
        "reported ERROR-level findings. Set GHG_FAIL_ON_DATA_INTEGRITY_ERROR=False "
        "to build it anyway."
    )
    emissions_ext_for_ghg = None
else:
    emissions_ext_for_ghg = emissions_ext


# Indicator definitions
# Uses the dynamically resolved extensions from the inspection cell above.

value_added_rows, value_added_note = choose_value_added_rows(
    factor_inputs_ext,
    override=VALUE_ADDED_ROW_SELECTION,
)

indicator_specs = {
    "economic_importance": {
        "extension": factor_inputs_ext,
        "row_selector": value_added_rows,
        "unit": detect_common_unit(factor_inputs_ext, value_added_rows),
        "label": "Value added",
        "definition": "factor_inputs",
    }
}

material_note = None
if materials_ext is not None:
    material_rows, material_note = choose_material_rows(materials_ext, override=MATERIAL_ROW_SELECTION)
    indicator_specs["materials"] = {
        "extension": materials_ext,
        "row_selector": material_rows,
        "unit": detect_common_unit(materials_ext, material_rows),
        "label": "Material footprint",
        "definition": materials_name,
    }
else:
    material_note = (
        "No dedicated material extension was resolved automatically. "
        "The notebook will continue without a material footprint result unless you manually map the right extension."
    )

ghg_note = None
ghg_row_selector = None
ghg_extension = None
ghg_definition = None
ghg_impact_extension = None
ghg_impact_row = None
ghg_proxy_extension = None
ghg_proxy_row_selector = None
ghg_proxy_note = None
ghg_row_factors = None

if impact_ext is not None:
    ghg_impact_row = choose_first_matching_row(
        impact_ext,
        exact=["global warming (GWP100)"],
        contains=["global warming (gwp100)", "global warming", "gwp100"],
    )
    ghg_impact_extension = impact_ext

if emissions_ext_for_ghg is not None:
    ghg_row_factors, ghg_note_detail = choose_ghg_row_factors(
        emissions_ext_for_ghg,
        override=GHG_ROW_SELECTION,
        default_factors=GWP100_FACTORS,
        include_biogenic=GHG_INCLUDE_BIOGENIC,
        include_precharacterized_eq_rows=GHG_INCLUDE_PRECHARACTERIZED_EQ_ROWS,
        include_sf6=GHG_INCLUDE_SF6,
    )
    base_mass_unit = detect_common_unit(emissions_ext_for_ghg, list(ghg_row_factors.keys())) or "mass unit"
    ghg_label = "GHG proxy (CO2+CH4+N2O, GWP100)"
    ghg_proxy_extension = build_characterized_proxy_extension(
        emissions_ext_for_ghg,
        row_factor_map=ghg_row_factors,
        row_name=ghg_label,
        impact_unit=f"{base_mass_unit} CO2-eq" if "CO2-eq" not in str(base_mass_unit) else str(base_mass_unit),
    )
    ghg_proxy_row_selector = ghg_label
    ghg_proxy_note = (
        "Built a transparent GHG proxy from raw emissions using the configured GWP100 factors. "
        f"{ghg_note_detail}"
    )

if GHG_FAIL_IF_ONLY_PROXY and ghg_impact_extension is None:
    ghg_note = (
        "No characterized impact extension was found and GHG_FAIL_IF_ONLY_PROXY=True, "
        "so GHG hotspot results are skipped."
    )
else:
    if ghg_impact_extension is not None and (
        GHG_PREFERRED_SOURCE == "impact" or ghg_proxy_extension is None
    ):
        ghg_extension = ghg_impact_extension
        ghg_row_selector = ghg_impact_row
        ghg_definition = f"characterized::{impact_name}"
        ghg_note = f"Using characterized GHG impacts from `{impact_name}` as the primary GHG definition."
    elif ghg_proxy_extension is not None:
        ghg_extension = ghg_proxy_extension
        ghg_row_selector = ghg_proxy_row_selector
        ghg_definition = "proxy::raw_emissions"
        ghg_note = (
            "No characterized impact extension was selected as primary, so the notebook uses the raw-emissions proxy "
            f"as the primary GHG definition. {ghg_proxy_note}"
        )
    else:
        ghg_note = (
            "Neither a characterized impact extension nor a raw emissions extension was found. "
            "GHG hotspot results are skipped."
        )

if ghg_extension is not None:
    indicator_specs["ghg"] = {
        "extension": ghg_extension,
        "row_selector": ghg_row_selector,
        "unit": detect_common_unit(ghg_extension, ghg_row_selector),
        "label": "GHG impacts" if ghg_definition and ghg_definition.startswith("characterized") else "GHG proxy (CO2+CH4+N2O)",
        "definition": ghg_definition,
    }

print(value_added_note)
print(material_note)
print(ghg_note)

if ghg_impact_extension is not None and ghg_proxy_extension is not None and GHG_COMPARE_PROXY_WHEN_IMPACT_AVAILABLE:
    print("Both a characterized impact definition and a raw-emissions proxy are available. The notebook will compare them later.")

print("Value-added row selection:")
if isinstance(value_added_rows, (list, pd.Index, np.ndarray)):
    display(pd.DataFrame({"selected_value_added_rows": [row_to_text(x) for x in value_added_rows]}))
else:
    display(pd.DataFrame({"selected_value_added_rows": [row_to_text(value_added_rows)]}))

if "materials" in indicator_specs:
    material_rows = indicator_specs["materials"]["row_selector"]
    print("Material row selection:")
    if isinstance(material_rows, (list, pd.Index, np.ndarray)):
        display(pd.DataFrame({"selected_material_rows": [row_to_text(x) for x in material_rows]}))
    else:
        display(pd.DataFrame({"selected_material_rows": [row_to_text(material_rows)]}))

if ghg_impact_extension is not None:
    print("Characterized GHG row selection:")
    display(pd.DataFrame({"selected_impact_rows": [row_to_text(ghg_impact_row)]}))

if ghg_proxy_extension is not None:
    print("GHG proxy row selection and factors:")
    display(
        pd.DataFrame(
            {
                "row": [row_to_text(r) for r in ghg_row_factors.keys()],
                "factor": list(ghg_row_factors.values()),
            }
        )
    )


In [ ]:

# Build Sweden hotspot rankings for CBA and PBA
ranking_tables = []

for indicator_name, spec in indicator_specs.items():
    ext = spec["extension"]
    row_selector = spec["row_selector"]
    unit = spec["unit"]
    definition = spec.get("definition")

    cba_df = ranking_from_manual_account(
        mrio=mrio,
        ext=ext,
        row_selector=row_selector,
        target_region=TARGET_REGION,
        account_name="CBA",
        indicator_name=indicator_name,
        Y_agg=Y_agg,
        unit=unit,
    )
    cba_df["definition"] = definition
    ranking_tables.append(cba_df)

    pba_df = ranking_from_manual_account(
        mrio=mrio,
        ext=ext,
        row_selector=row_selector,
        target_region=TARGET_REGION,
        account_name="PBA",
        indicator_name=indicator_name,
        Y_agg=Y_agg,
        unit=unit,
    )
    pba_df["definition"] = definition
    ranking_tables.append(pba_df)

rankings = pd.concat(ranking_tables, ignore_index=True)
rankings.to_csv(OUTPUT_DIR / "sweden_sector_rankings_tidy.csv", index=False)

display(rankings.head(20))
print(f"Saved: {OUTPUT_DIR / 'sweden_sector_rankings_tidy.csv'}")


In [ ]:
# Display top sectors by indicator and accounting principle
for account_name in ["CBA", "PBA"]:
    for indicator_name, spec in indicator_specs.items():
        title = f"{account_name} hotspots for Sweden – {spec['label']}"
        subset = (
            rankings
            .query("account == @account_name and indicator == @indicator_name")
            .sort_values("rank")
            .head(TOP_N_SECTORS)
            .reset_index(drop=True)
        )
        display(subset)
        plot_top_sectors(subset, title=title, top_n=TOP_N_SECTORS)



In [ ]:

# Internal consistency checks and comparison against extension-stored accounts where available
ranking_checks = []
extension_vs_manual_checks = []
ghg_definition_comparison_summary = pd.DataFrame()
ghg_definition_comparison_details = pd.DataFrame()
ghg_proxy_rows_df = pd.DataFrame()
ghg_core_rows_df = pd.DataFrame()
ghg_sanity_df = pd.DataFrame()

for indicator_name, spec in indicator_specs.items():
    ext = spec["extension"]
    row_selector = spec["row_selector"]

    cba_total_rank = (
        rankings
        .query("account == 'CBA' and indicator == @indicator_name")
        ["value"]
        .sum()
    )
    pba_total_rank = (
        rankings
        .query("account == 'PBA' and indicator == @indicator_name")
        ["value"]
        .sum()
    )

    cba_total_manual = float(manual_cba_sector_vector(mrio, ext, row_selector, TARGET_REGION, Y_agg).sum())
    pba_total_manual = float(manual_pba_sector_vector(mrio, ext, row_selector, TARGET_REGION).sum())

    ranking_checks.append({
        "indicator": indicator_name,
        "account": "CBA",
        "ranking_sum": cba_total_rank,
        "manual_model_sum": cba_total_manual,
        "difference": cba_total_rank - cba_total_manual,
    })
    ranking_checks.append({
        "indicator": indicator_name,
        "account": "PBA",
        "ranking_sum": pba_total_rank,
        "manual_model_sum": pba_total_manual,
        "difference": pba_total_rank - pba_total_manual,
    })

    cmp_df = compare_extension_and_manual_accounts(
        mrio=mrio,
        ext=ext,
        row_selector=row_selector,
        target_region=TARGET_REGION,
        Y_agg=Y_agg,
        indicator_name=indicator_name,
    )
    if not cmp_df.empty:
        extension_vs_manual_checks.append(cmp_df)

ranking_checks = pd.DataFrame(ranking_checks)
display(ranking_checks)

if extension_vs_manual_checks:
    extension_vs_manual_checks = pd.concat(extension_vs_manual_checks, ignore_index=True)
    print("Manual accounts versus extension-stored accounts:")
    display(extension_vs_manual_checks)
else:
    extension_vs_manual_checks = pd.DataFrame()

if "ghg" in indicator_specs:
    ghg_sanity_df = ghg_sanity_check_table(
        rankings,
        indicator_key="ghg",
        suspicious_ratio=GHG_SUSPICIOUS_CBA_TO_PBA_RATIO,
    )
    print("GHG sanity check:")
    display(ghg_sanity_df)

if ghg_proxy_extension is not None and ghg_row_factors is not None:
    ghg_proxy_rows_df = ghg_proxy_row_diagnostics(
        mrio=mrio,
        emissions_ext=(emissions_ext_for_ghg if 'emissions_ext_for_ghg' in globals() and emissions_ext_for_ghg is not None else emissions_ext),
        row_factor_map=ghg_row_factors,
        target_region=TARGET_REGION,
        Y_agg=Y_agg,
    )
    print("GHG proxy totals by selected raw-emission row:")
    display(ghg_proxy_rows_df)

    ghg_core_rows_df = ghg_core_row_check_table(
        mrio=mrio,
        emissions_ext=(emissions_ext_for_ghg if 'emissions_ext_for_ghg' in globals() and emissions_ext_for_ghg is not None else emissions_ext),
        row_factor_map=ghg_row_factors,
        target_region=TARGET_REGION,
        Y_agg=Y_agg,
        core_rows=GHG_CORE_DIAGNOSTIC_ROWS,
    )
    print("GHG core-row diagnostic check:")
    display(ghg_core_rows_df)

if ghg_impact_extension is not None and ghg_proxy_extension is not None and GHG_COMPARE_PROXY_WHEN_IMPACT_AVAILABLE:
    impact_cba = manual_cba_sector_vector(mrio, ghg_impact_extension, ghg_impact_row, TARGET_REGION, Y_agg)
    impact_pba = manual_pba_sector_vector(mrio, ghg_impact_extension, ghg_impact_row, TARGET_REGION)
    proxy_cba = manual_cba_sector_vector(mrio, ghg_proxy_extension, ghg_proxy_row_selector, TARGET_REGION, Y_agg)
    proxy_pba = manual_pba_sector_vector(mrio, ghg_proxy_extension, ghg_proxy_row_selector, TARGET_REGION)

    cba_summary, cba_details = compare_two_sector_vectors(
        impact_cba,
        proxy_cba,
        label_a="impact",
        label_b="proxy",
        top_n=TOP_N_SECTORS,
    )
    cba_summary["account"] = "CBA"
    pba_summary, pba_details = compare_two_sector_vectors(
        impact_pba,
        proxy_pba,
        label_a="impact",
        label_b="proxy",
        top_n=TOP_N_SECTORS,
    )
    pba_summary["account"] = "PBA"

    ghg_definition_comparison_summary = pd.concat([cba_summary, pba_summary], ignore_index=True)
    cba_details["account"] = "CBA"
    pba_details["account"] = "PBA"
    ghg_definition_comparison_details = pd.concat([cba_details, pba_details], ignore_index=True)

    print("GHG definition comparison: characterized impact versus raw-emissions proxy")
    display(ghg_definition_comparison_summary)

    print("GHG definition comparison details (top rows sorted by proxy value)")
    display(ghg_definition_comparison_details.head(30))

ghg_sanity_df.to_csv(OUTPUT_DIR / "ghg_sanity_check.csv", index=False)
ghg_proxy_rows_df.to_csv(OUTPUT_DIR / "ghg_proxy_row_diagnostics.csv", index=False)
ghg_core_rows_df.to_csv(OUTPUT_DIR / "ghg_core_row_diagnostics.csv", index=False)
ghg_definition_comparison_summary.to_csv(OUTPUT_DIR / "ghg_definition_comparison_summary.csv", index=False)
ghg_definition_comparison_details.to_csv(OUTPUT_DIR / "ghg_definition_comparison_details.csv", index=False)


In [ ]:
# Build country breakdowns for hotspot sectors
cba_source_tables_full = []
cba_source_tables_shown = []
pba_destination_tables_full = []
pba_destination_tables_shown = []

for indicator_name, spec in indicator_specs.items():
    ext = spec["extension"]
    row_selector = spec["row_selector"]
    unit = spec["unit"]

    cba_hotspots = (
        rankings
        .query("account == 'CBA' and indicator == @indicator_name")
        .sort_values("rank")
        .head(TOP_N_SOURCE_SECTORS)
        .copy()
    )

    for _, row in cba_hotspots.iterrows():
        sector = row["sector"]
        sector_value = row["value"]

        src_full, src_shown = cba_source_country_shares(
            mrio=mrio,
            ext=ext,
            row_selector=row_selector,
            target_region=TARGET_REGION,
            target_sector=sector,
            Y_agg=Y_agg,
            top_n=TOP_N_SOURCE_COUNTRIES,
        )

        for df in [src_full, src_shown]:
            df["indicator"] = indicator_name
            df["target_sector_value"] = sector_value
            df["unit"] = unit

        cba_source_tables_full.append(src_full)
        cba_source_tables_shown.append(src_shown)

    pba_hotspots = (
        rankings
        .query("account == 'PBA' and indicator == @indicator_name")
        .sort_values("rank")
        .head(TOP_N_SOURCE_SECTORS)
        .copy()
    )

    for _, row in pba_hotspots.iterrows():
        sector = row["sector"]
        sector_value = row["value"]

        dest_full, dest_shown = pba_destination_country_shares(
            mrio=mrio,
            ext=ext,
            row_selector=row_selector,
            source_region=TARGET_REGION,
            source_sector=sector,
            Y_agg=Y_agg,
            top_n=TOP_N_DESTINATION_COUNTRIES,
        )

        for df in [dest_full, dest_shown]:
            df["indicator"] = indicator_name
            df["source_sector_value"] = sector_value
            df["unit"] = unit

        pba_destination_tables_full.append(dest_full)
        pba_destination_tables_shown.append(dest_shown)

cba_source_country_shares_full_df = pd.concat(cba_source_tables_full, ignore_index=True)
cba_source_country_shares_df = pd.concat(cba_source_tables_shown, ignore_index=True)
pba_destination_country_shares_full_df = pd.concat(pba_destination_tables_full, ignore_index=True)
pba_destination_country_shares_df = pd.concat(pba_destination_tables_shown, ignore_index=True)

cba_breakdown_checks = summarise_breakdown_quality(
    cba_source_country_shares_full_df,
    group_cols=["indicator", "target_region", "target_sector"],
    value_col="source_value",
    total_col="model_total",
    reference_total_col="target_sector_value",
    share_col="source_share",
).sort_values(["indicator", "target_sector"])

pba_destination_checks = summarise_breakdown_quality(
    pba_destination_country_shares_full_df,
    group_cols=["indicator", "source_region", "source_sector"],
    value_col="destination_value",
    total_col="model_total",
    reference_total_col="source_sector_value",
    share_col="destination_share",
).sort_values(["indicator", "source_sector"])

cba_source_country_shares_df.to_csv(OUTPUT_DIR / "sweden_cba_source_country_shares_top.csv", index=False)
cba_source_country_shares_full_df.to_csv(OUTPUT_DIR / "sweden_cba_source_country_shares_full.csv", index=False)
pba_destination_country_shares_df.to_csv(OUTPUT_DIR / "sweden_pba_destination_country_shares_top.csv", index=False)
pba_destination_country_shares_full_df.to_csv(OUTPUT_DIR / "sweden_pba_destination_country_shares_full.csv", index=False)
cba_breakdown_checks.to_csv(OUTPUT_DIR / "sweden_cba_source_country_checks.csv", index=False)
pba_destination_checks.to_csv(OUTPUT_DIR / "sweden_pba_destination_country_checks.csv", index=False)

print("CBA source-country shares (shown table, first 30 rows):")
display(cba_source_country_shares_df.head(30))

print("CBA source-country closure checks (based on full tables):")
display(cba_breakdown_checks)

print("PBA destination-country shares (final-demand destinations; shown table, first 30 rows):")
display(pba_destination_country_shares_df.head(30))

print("PBA final-demand destination-country closure checks (based on full tables):")
display(pba_destination_checks)

print(f"Saved: {OUTPUT_DIR / 'sweden_cba_source_country_shares_top.csv'}")
print(f"Saved: {OUTPUT_DIR / 'sweden_cba_source_country_shares_full.csv'}")
print(f"Saved: {OUTPUT_DIR / 'sweden_pba_destination_country_shares_top.csv'}")
print(f"Saved: {OUTPUT_DIR / 'sweden_pba_destination_country_shares_full.csv'}")


In [ ]:
# Optional: compact wide-format exports for easy browsing in Excel
rankings_wide = (
    rankings
    .pivot_table(
        index=["region", "sector"],
        columns=["account", "indicator"],
        values="value",
        aggfunc="first",
    )
    .sort_index(axis=1)
)

cba_source_shares_wide = (
    cba_source_country_shares_df
    .pivot_table(
        index=["indicator", "target_region", "target_sector"],
        columns="source_country",
        values="source_share",
        aggfunc="first",
    )
    .sort_index(axis=1)
)

pba_destination_shares_wide = (
    pba_destination_country_shares_df
    .pivot_table(
        index=["indicator", "source_region", "source_sector"],
        columns="destination_country",
        values="destination_share",
        aggfunc="first",
    )
    .sort_index(axis=1)
)

with pd.ExcelWriter(OUTPUT_DIR / "sweden_hotspot_analysis.xlsx", engine="openpyxl") as writer:
    rankings.to_excel(writer, sheet_name="rankings_tidy", index=False)
    rankings_wide.to_excel(writer, sheet_name="rankings_wide")
    cba_source_country_shares_df.to_excel(writer, sheet_name="cba_source_shares", index=False)
    cba_source_shares_wide.to_excel(writer, sheet_name="cba_source_shares_wide")
    cba_breakdown_checks.to_excel(writer, sheet_name="cba_source_checks", index=False)
    pba_destination_country_shares_df.to_excel(writer, sheet_name="pba_dest_shares", index=False)
    pba_destination_shares_wide.to_excel(writer, sheet_name="pba_dest_shares_wide")
    pba_destination_checks.to_excel(writer, sheet_name="pba_dest_checks", index=False)

rankings.to_csv(OUTPUT_DIR / "sweden_sector_rankings_tidy.csv", index=False)
print(f"Saved: {OUTPUT_DIR / 'sweden_hotspot_analysis.xlsx'}")



# Additional GHG diagnostics
with pd.ExcelWriter(OUTPUT_DIR / "sweden_hotspot_analysis.xlsx", engine="openpyxl", mode="a", if_sheet_exists="replace") as writer:
    ghg_sanity_df.to_excel(writer, sheet_name="ghg_sanity_check", index=False)
    ghg_proxy_rows_df.to_excel(writer, sheet_name="ghg_proxy_rows", index=False)
    ghg_definition_comparison_summary.to_excel(writer, sheet_name="ghg_def_compare_sum", index=False)
    ghg_definition_comparison_details.to_excel(writer, sheet_name="ghg_def_compare_det", index=False)
print(f"Updated workbook with GHG diagnostics: {OUTPUT_DIR / 'sweden_hotspot_analysis.xlsx'}")


# Notes for interpretation

- **CBA rankings** show which Swedish final-demand sectors are associated with the largest global value added, GHG burden, or material footprint.
- **PBA rankings** show which Swedish production sectors account for the largest direct impacts occurring in Sweden.
- **CBA source-country shares** trace the hotspot sector back to the countries where the direct value added / emissions / material extraction occur in the supply chain.
- **PBA destination-country shares** allocate a Swedish production hotspot across the destination countries whose final demand induces that Swedish output.
- For **PBA source country**, the answer is trivially Sweden = 100%; the destination breakdown is more informative, so it is reported explicitly.
- The notebook now includes reconstruction checks so you can see whether the country decomposition reproduces the corresponding model total.
- If the notebook is rerun after an error, restart the kernel and run from the top to avoid stale outputs remaining in later cells.


- **PBA destination-country tables** allocate Sweden's *direct* production-based burdens to the countries whose **final demand** ultimately drives that Swedish output. They are not the same as direct export-partner statistics.
- **Closure checks** should be read with the pass/fail columns. Tiny residuals around machine precision are numerical noise, not substantive inconsistencies.

- **GHG results** are only directly comparable across runs when they use the same definition. If one run uses a characterized `impacts` row and another uses the raw-emissions proxy, compare them only after inspecting the new GHG diagnostic tables.


## Files written by this notebook

The notebook writes the following files to `OUTPUT_DIR`:

- `sweden_sector_rankings_tidy.csv`
- `sweden_cba_source_country_shares.csv`
- `sweden_pba_destination_country_shares.csv`
- `sweden_cba_source_country_checks.csv`
- `sweden_pba_destination_country_checks.csv`
- `sweden_hotspot_analysis.xlsx`
